# Computational Modeling of Infant Auditory Development
## Capstone Project
### Using AudioCaps as a Naturalistic Audio Proxy

**Background**: The Science paper *"Grounded language acquisition through the eyes and ears of a single child"* (Orhan et al., 2024) demonstrated that deep neural networks trained on a single infant's naturalistic audiovisual experience (SAYCam) can develop structured perceptual representations. This project computationally probes the same question for the **auditory modality** using AudioCaps as our naturalistic audio corpus.

**Research Questions (from proposal)**:
- **RQ1**: Can models learn auditory structure (speech/non-speech, sound categories) without explicit labels?
- **RQ2**: How does the mix of sounds shape learned representations?
- **RQ3**: Do early auditory skills emerge before later ones (developmental milestones)?
- **RQ4**: Do model representations align with infant auditory perception findings?

**Pipeline**: AudioCaps → Weak label generation → Feature extraction (MFCC + pretrained AST) → Classification → Visualization → Developmental analysis

In [ ]:
import os, sys
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
sys.path.insert(0, os.path.abspath("scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import librosa
import soundfile as sf
import torch
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
import umap
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DATA_DIR  = Path("data")
FEAT_DIR  = DATA_DIR / "features"
AUDIO_DIR = DATA_DIR / "audio"
FIG_DIR   = Path("results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Section 1: Dataset Overview
### AudioCaps — A Naturalistic Audio Corpus
AudioCaps contains ~57K audio clips sourced from AudioSet (YouTube), with human-written captions describing the sounds. This diverse corpus includes speech, music, nature sounds, and environmental noise — analogous to the varied soundscape an infant encounters daily.

In [ ]:
# Load labeled metadata
labeled = pd.read_csv(DATA_DIR / "audiocaps_labeled.csv")
print(f"Total clips: {len(labeled)}")
print(f"\nColumns: {list(labeled.columns)}")
print(f"\nSample captions:")
print(labeled[["caption" if "caption" in labeled.columns else "answer", "category", "is_speech"]].head(8).to_string())

# Category distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cat_col = "category"
counts = labeled[cat_col].value_counts()
COLORS = {"speech": "#E74C3C", "music": "#3498DB", "nature": "#2ECC71",
           "environment": "#F39C12", "animal": "#9B59B6"}
colors = [COLORS.get(c, "gray") for c in counts.index]

axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white")
axes[0].set_title("Sound Category Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=30)

speech_counts = labeled["is_speech"].value_counts()
axes[1].pie([speech_counts.get(1, 0), speech_counts.get(0, 0)],
            labels=["Speech", "Non-speech"],
            colors=["#E74C3C", "#3498DB"], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Speech vs. Non-speech Ratio")

plt.suptitle("AudioCaps Dataset Distribution (Weakly Labeled via Captions)", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "dataset_distribution.png", bbox_inches="tight")
plt.show()

## Section 2: Feature Extraction
### Two levels of representation
1. **MFCCs** — classical acoustic fingerprint (40 coefficients, mean+std pooled → 80-dim). Captures spectral shape, widely used in infant speech research.
2. **Pretrained AST embeddings** — Audio Spectrogram Transformer finetuned on AudioSet (MIT/ast-finetuned-audioset-10-10-0.4593, 768-dim). Captures high-level semantic audio features.

This comparison directly addresses **RQ4**: does a pretrained model capture structure aligned with known infant auditory sensitivities?

In [ ]:
# Load pre-extracted features (run 03_feature_extraction.py first)
mfcc_feats = np.load(FEAT_DIR / "mfcc.npy")
logmel_feats = np.load(FEAT_DIR / "logmel.npy")
ast_feats = np.load(FEAT_DIR / "ast_embeddings.npy")

n = len(ast_feats)
labels_cat = labeled["category"].values[:n]
labels_speech = labeled["is_speech"].values[:n]

print(f"Audio clips with features: {n}")
print(f"MFCC shape:   {mfcc_feats.shape}  (80-dim: 40 mean + 40 std)")
print(f"Log-mel shape:{logmel_feats.shape} (128-dim: 64 mean + 64 std)")
print(f"AST shape:    {ast_feats.shape}  (768-dim transformer hidden states)")

# Show a sample MFCC
wav_files = sorted(AUDIO_DIR.glob("clip_*.wav"))
if wav_files:
    audio, sr = librosa.load(str(wav_files[0]), sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    fig, axes = plt.subplots(1, 2, figsize=(13, 3))
    librosa.display.waveshow(audio, sr=sr, ax=axes[0])
    axes[0].set_title("Waveform (example clip)")
    img = librosa.display.specshow(mfcc, sr=sr, x_axis="time", ax=axes[1], cmap="viridis")
    plt.colorbar(img, ax=axes[1])
    axes[1].set_title("MFCC (40 coefficients)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "sample_mfcc.png", bbox_inches="tight")
    plt.show()
else:
    print("(No audio files yet — run 01_download_data.py and 03_feature_extraction.py)")

## Section 3: Auditory Capability Evaluation (RQ1)
### Can features distinguish speech from non-speech and categorize sounds?
We train a logistic regression classifier on each feature type with 5-fold cross-validation. This mimics a model that has learned from naturalistic audio and is probed for auditory understanding — analogous to behavioral tests of infant auditory discrimination.

In [ ]:
def evaluate_feature(X, y_raw, task_name, feature_name, cv=5):
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    accs, f1s = [], []
    pipe = Pipeline([("sc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42))])
    for tr, val in skf.split(X, y):
        pipe.fit(X[tr], y[tr])
        preds = pipe.predict(X[val])
        accs.append(accuracy_score(y[val], preds))
        f1s.append(f1_score(y[val], preds, average="weighted"))
    return {"feature": feature_name, "task": task_name,
            "acc": np.mean(accs), "acc_std": np.std(accs),
            "f1": np.mean(f1s), "f1_std": np.std(f1s)}

feature_sets = {"MFCC (80-dim)": mfcc_feats, "Log-mel (128-dim)": logmel_feats,
                "AST pretrained (768-dim)": ast_feats}

results = []
for fname, X in feature_sets.items():
    r1 = evaluate_feature(X, labels_speech, "Speech vs. Non-speech", fname)
    r2 = evaluate_feature(X, labels_cat,    "Sound Categorization",  fname)
    results.extend([r1, r2])
    print(f"[{fname}] Speech: Acc={r1['acc']:.3f}±{r1['acc_std']:.3f} | "
          f"Category: Acc={r2['acc']:.3f}±{r2['acc_std']:.3f}")

df_res = pd.DataFrame(results)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bar_colors = ["#95A5A6", "#F39C12", "#E74C3C"]

for ax, task in zip(axes, ["Speech vs. Non-speech", "Sound Categorization"]):
    sub = df_res[df_res["task"] == task].reset_index(drop=True)
    x = np.arange(len(sub))
    bars = ax.bar(x - 0.2, sub["acc"], 0.35, yerr=sub["acc_std"],
                  capsize=4, color=bar_colors, label="Accuracy", alpha=0.85)
    bars2 = ax.bar(x + 0.2, sub["f1"], 0.35, yerr=sub["f1_std"],
                   capsize=4, color=bar_colors, label="F1 Score",
                   alpha=0.5, hatch="//", edgecolor="black")
    ax.set_xticks(x)
    ax.set_xticklabels(sub["feature"], rotation=15, ha="right", fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.set_title(task)
    if ax == axes[0]:
        ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.axhline(0.5, color="gray", linestyle="--", alpha=0.4, label="Chance (binary)")

plt.suptitle("RQ1: Auditory Capability Evaluation\n"
             "(Logistic Regression, 5-fold CV)", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "classification_results.png", bbox_inches="tight")
plt.show()
df_res.round(3)

## Section 4: Representation Visualization (Proposal §7)
### PCA, t-SNE, and UMAP of audio embeddings
Do representations cluster by meaningful auditory categories? Infant perceptual research finds early sensitivity to speech prosody and later differentiation of finer categories — we look for analogous structure in the learned embeddings.

In [ ]:
CAT_COLORS = {"speech": "#E74C3C", "music": "#3498DB", "nature": "#2ECC71",
              "environment": "#F39C12", "animal": "#9B59B6"}

def scatter_2d(ax, coords, labels, title, colors_map):
    for lbl in sorted(set(labels)):
        mask = labels == lbl
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=colors_map.get(str(lbl), "gray"), label=str(lbl),
                   alpha=0.5, s=10, edgecolors="none")
    ax.legend(markerscale=3, fontsize=8, loc="best")
    ax.set_title(title, fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

scaler = StandardScaler()
ast_sc = scaler.fit_transform(ast_feats)
mfcc_sc = StandardScaler().fit_transform(mfcc_feats)

# PCA
pca2 = PCA(n_components=2, random_state=42)
ast_pca  = pca2.fit_transform(ast_sc)
mfcc_pca = PCA(n_components=2, random_state=42).fit_transform(mfcc_sc)

# t-SNE on PCA-reduced AST
pca50 = PCA(n_components=50, random_state=42).fit_transform(ast_sc)
ast_tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42).fit_transform(pca50)

# UMAP
ast_umap = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(ast_sc)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
speech_color_map = {"speech": "#E74C3C", "non-speech": "#3498DB"}
speech_labels = np.array(["speech" if s else "non-speech" for s in labels_speech])

scatter_2d(axes[0, 0], mfcc_pca,  labels_cat,    "PCA — MFCC (Categories)",    CAT_COLORS)
scatter_2d(axes[0, 1], ast_pca,   labels_cat,    "PCA — AST (Categories)",     CAT_COLORS)
scatter_2d(axes[1, 0], ast_tsne,  labels_cat,    "t-SNE — AST (Categories)",   CAT_COLORS)
scatter_2d(axes[1, 1], ast_umap,  speech_labels, "UMAP — AST (Speech vs. Non-speech)", speech_color_map)

plt.suptitle("Auditory Representation Space\n"
             "Structure in AST embeddings reflects meaningful sound categories", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "representation_grid.png", bbox_inches="tight")
plt.show()

# PCA explained variance
pca_full = PCA(random_state=42).fit(ast_sc)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, 51), cumvar[:50], "b-o", markersize=4)
ax.axhline(0.9, color="r", linestyle="--", label="90% variance")
ax.set_xlabel("PCA Components"); ax.set_ylabel("Cumulative Variance")
ax.set_title("PCA: How many dimensions capture AST structure?")
ax.legend(); ax.grid(alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_variance.png", bbox_inches="tight")
plt.show()
print(f"Components needed for 90% variance: {np.argmax(cumvar >= 0.90) + 1}")

## Section 5: Developmental Trajectory Analysis (RQ3)
### Do some auditory skills emerge before others?
We simulate "experience" by training on increasing fractions of the data and tracking how performance on different tasks evolves. This mirrors developmental research showing infants acquire broad auditory categories (speech/non-speech) before finer distinctions.

**Hypothesis**: Speech detection should plateau earlier than multi-class categorization, analogous to infants' early preference for speech over non-speech.

In [ ]:
fractions = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
n_test = int(n * 0.2)
X_pool, X_test = ast_feats[:-n_test], ast_feats[-n_test:]
y_sp_pool, y_sp_test = labels_speech[:-n_test], labels_speech[-n_test:]
y_cat_pool, y_cat_test = labels_cat[:-n_test], labels_cat[-n_test:]

def lc_task(X_pool, y_pool_raw, X_test, y_test_raw, fracs):
    le = LabelEncoder().fit(np.concatenate([y_pool_raw, y_test_raw]))
    y_pool = le.transform(y_pool_raw)
    y_test = le.transform(y_test_raw)
    accs = []
    for frac in fracs:
        n_tr = max(int(len(X_pool) * frac), 20)
        pipe = Pipeline([("sc", StandardScaler()),
                         ("clf", LogisticRegression(max_iter=500, random_state=42))])
        pipe.fit(X_pool[:n_tr], y_pool[:n_tr])
        accs.append(accuracy_score(y_test, pipe.predict(X_test)))
    return accs

sp_accs  = lc_task(X_pool, y_sp_pool,  X_test, y_sp_test,  fractions)
cat_accs = lc_task(X_pool, y_cat_pool, X_test, y_cat_test, fractions)

# Milestone: first fraction to exceed threshold
THRESHOLD = 0.70
def milestone(accs, fracs, thresh=THRESHOLD):
    for f, a in zip(fracs, accs):
        if a >= thresh:
            return f
    return None

ms_sp  = milestone(sp_accs,  fractions)
ms_cat = milestone(cat_accs, fractions)

x_pct = [f * 100 for f in fractions]
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_pct, sp_accs,  "r-o", linewidth=2, markersize=7, label="Speech vs. Non-speech")
ax.plot(x_pct, cat_accs, "b-s", linewidth=2, markersize=7, label="Sound Categorization (5-class)")
ax.axhline(THRESHOLD, color="gray", linestyle="--", alpha=0.6, label=f"{THRESHOLD:.0%} milestone")
if ms_sp:
    ax.axvline(ms_sp * 100, color="r", linestyle=":", alpha=0.5)
    ax.annotate(f"Speech milestone\n@ {ms_sp:.0%} data",
                xy=(ms_sp*100, THRESHOLD), xytext=(ms_sp*100+5, THRESHOLD-0.08),
                fontsize=9, color="r", arrowprops=dict(arrowstyle="->", color="r"))
if ms_cat:
    ax.axvline(ms_cat * 100, color="b", linestyle=":", alpha=0.5)
    ax.annotate(f"Category milestone\n@ {ms_cat:.0%} data",
                xy=(ms_cat*100, THRESHOLD), xytext=(ms_cat*100+5, THRESHOLD+0.03),
                fontsize=9, color="b", arrowprops=dict(arrowstyle="->", color="b"))

ax.set_xlabel("Training Data Used (%)")
ax.set_ylabel("Test Accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("RQ3: Developmental Trajectory of Auditory Capabilities\n"
             "(AST embeddings — simulated experience)", fontsize=11)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "developmental_trajectory.png", bbox_inches="tight")
plt.show()

print(f"Speech detection reaches {THRESHOLD:.0%} accuracy at {ms_sp:.0%} of training data")
print(f"Sound categorization reaches {THRESHOLD:.0%} accuracy at {ms_cat:.0%} of training data")
print(f"\nInterpretation: {'Speech detection emerges EARLIER' if ms_sp < ms_cat else 'Both emerge simultaneously'}"
      f" — {'consistent' if ms_sp and ms_cat and ms_sp < ms_cat else 'not consistent'} "
      f"with infant developmental patterns")

## Section 6: Input Sensitivity Analysis (RQ2)
### How does the training sound environment shape representations?
We train probes on three curated subsets mimicking different acoustic environments an infant might inhabit, then evaluate on a balanced held-out set. This directly addresses RQ2: does a speech-rich environment produce better speech detectors?

In [ ]:
np.random.seed(42)
speech_idx    = np.where(labels_speech == 1)[0]
nonspeech_idx = np.where(labels_speech == 0)[0]

# Fixed balanced test set
all_idx = np.arange(n)
np.random.shuffle(all_idx)
n_test_small = min(500, n // 5)
test_idx = all_idx[:n_test_small]
X_test_small = ast_feats[test_idx]
y_test_sp  = labels_speech[test_idx]
y_test_cat = labels_cat[test_idx]

N_TRAIN = min(1000, n - n_test_small)

def sample_regime(speech_frac, n_total):
    n_sp  = int(n_total * speech_frac)
    n_nsp = n_total - n_sp
    sp  = np.random.choice(speech_idx,    min(n_sp,  len(speech_idx)),    replace=False)
    nsp = np.random.choice(nonspeech_idx, min(n_nsp, len(nonspeech_idx)), replace=False)
    return np.concatenate([sp, nsp])

regimes = {
    "Speech-\ndominant\n(80%)":    sample_regime(0.8, N_TRAIN),
    "Balanced\n(50%)":              sample_regime(0.5, N_TRAIN),
    "Non-speech-\ndominant\n(20%)": sample_regime(0.2, N_TRAIN),
}

regime_results = []
for rname, ridx in regimes.items():
    X_tr = ast_feats[ridx]
    y_tr_sp  = labels_speech[ridx]
    y_tr_cat = labels_cat[ridx]

    le_sp = LabelEncoder().fit(np.concatenate([y_tr_sp, y_test_sp]))
    pipe_sp = Pipeline([("sc", StandardScaler()),
                        ("clf", LogisticRegression(max_iter=500, random_state=42))])
    pipe_sp.fit(X_tr, le_sp.transform(y_tr_sp))
    acc_sp = accuracy_score(le_sp.transform(y_test_sp), pipe_sp.predict(X_test_small))
    f1_sp  = f1_score(le_sp.transform(y_test_sp), pipe_sp.predict(X_test_small), average="weighted")

    le_cat = LabelEncoder().fit(np.concatenate([y_tr_cat, y_test_cat]))
    pipe_cat = Pipeline([("sc", StandardScaler()),
                         ("clf", LogisticRegression(max_iter=500, random_state=42))])
    pipe_cat.fit(X_tr, le_cat.transform(y_tr_cat))
    acc_cat = accuracy_score(le_cat.transform(y_test_cat), pipe_cat.predict(X_test_small))
    f1_cat  = f1_score(le_cat.transform(y_test_cat), pipe_cat.predict(X_test_small), average="weighted")

    regime_results.append({"regime": rname, "acc_sp": acc_sp, "f1_sp": f1_sp,
                            "acc_cat": acc_cat, "f1_cat": f1_cat})
    print(f"[{rname.replace(chr(10),' ')}] Speech Acc={acc_sp:.3f} | Cat Acc={acc_cat:.3f}")

df_reg = pd.DataFrame(regime_results)
labels_r = df_reg["regime"].tolist()
x = np.arange(len(labels_r))
regime_colors = ["#E74C3C", "#3498DB", "#2ECC71"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric, task_title in [
    (axes[0], "sp",  "Speech vs. Non-speech Detection"),
    (axes[1], "cat", "Sound Categorization (5-class)")
]:
    accs = df_reg[f"acc_{metric}"].values
    f1s  = df_reg[f"f1_{metric}"].values
    ax.bar(x - 0.2, accs, 0.35, color=regime_colors, alpha=0.85, label="Accuracy")
    ax.bar(x + 0.2, f1s,  0.35, color=regime_colors, alpha=0.5, hatch="//",
           edgecolor="black", label="F1")
    ax.set_xticks(x); ax.set_xticklabels(labels_r, fontsize=9)
    ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
    ax.set_title(task_title)
    if ax == axes[0]: ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("RQ2: Impact of Training Sound Environment on Auditory Capabilities\n"
             "(Trained on regime subset, tested on balanced held-out set)", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "input_sensitivity.png", bbox_inches="tight")
plt.show()

## Section 7: Bayesian NMF — Discovering Sound Components

### What sub-structure does the AST model encode?

Our previous analysis used 5 hand-defined categories (speech, environment, nature, animal, music). But the AST model may encode **finer-grained distinctions** that cut across those labels. We apply **Bayesian Non-negative Matrix Factorization (BNMF)** to discover them automatically — with no label supervision.

**Adapted from**: *Meenakshi's ComponentAnalysis_BNMF_Tutorial* — originally designed for fMRI brain data (decomposing voxel responses into neural populations). The analogy maps directly to audio:

| fMRI (original tutorial) | Audio (this project) |
|---|---|
| Voxel responses (N images × D voxels) | AST embeddings (1442 clips × 768 features) |
| Neural population types | Sound-concept components |
| fMRI subject | Full AudioCaps corpus |
| Top stimuli per component | Top audio clips per component |

**Decomposition**:

$$V_{1442 \times 768} \;\approx\; W_{1442 \times K} \;\times\; H_{K \times 768}$$

- **W** (`cons_resp`): how much each audio clip activates each component  
- **H** (`cons_weights`): which AST feature dimensions define each component  
- **K = 10**: selected automatically via BIC (Bayesian Information Criterion)

The **consensus procedure** runs 20 independent BNMF trials and aggregates stable components (density-based outlier filtering + K-Means clustering), making results robust to random initialization.

In [ ]:
import seaborn as sns

BNMF_DIR = Path("results/bnmf")
cons_resp    = np.load(BNMF_DIR / "consensus_response.npy")  # (1442, K)
cons_weights = np.load(BNMF_DIR / "consensus_weights.npy")  # (K, 768)
bics         = np.load(BNMF_DIR / "bics.npy")

actual_k    = cons_resp.shape[1]
search_list = list(range(5, 26, 5))
optimal_k   = search_list[int(np.argmin(bics))]

COMP_LABELS = {
    1: "Railway",      2: "Machinery",    3: "Water/Rain",
    4: "Crowd noise",  5: "Impact sounds",6: "Infant/Animal",
    7: "Female speech",8: "Weather/Waves",9: "Outdoor mixed", 10: "Vehicles",
}
BNMF_COLORS = [plt.cm.tab10(i / 10) for i in range(actual_k)]

print(f"Consensus response : {cons_resp.shape}  (clips × K components)")
print(f"Consensus weights  : {cons_weights.shape}  (K components × AST features)")
print(f"Optimal K = {optimal_k}  (selected via BIC)\n")
print("Discovered components:")
for c in range(actual_k):
    top_cat = pd.Series(labels_cat[np.argsort(cons_resp[:, c])[::-1][:50]]).value_counts().index[0]
    print(f"  C{c+1:2d}: {COMP_LABELS.get(c+1,'?'):<20s}  (dominant category in top-50: {top_cat})")

# BIC curve
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(search_list, bics, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(optimal_k, color='red', linestyle='--', linewidth=1.5,
           label=f'Optimal K = {optimal_k}')
ax.set_xlabel('Number of Components (K)')
ax.set_ylabel('BIC (lower = better fit)')
ax.set_title('BIC Criterion — Automatic Component Selection')
ax.legend(); ax.grid(alpha=0.25)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "bnmf_bic.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Component × Category heatmap ──────────────────────────────────────────────
ALL_CATS = ['speech', 'environment', 'nature', 'animal', 'music']

heatmap_data = np.zeros((actual_k, len(ALL_CATS)))
for j, cat in enumerate(ALL_CATS):
    idx = np.where(labels_cat == cat)[0]
    if len(idx):
        heatmap_data[:, j] = cons_resp[idx].mean(0)

yticklabels = [f'C{i+1}: {COMP_LABELS.get(i+1, "")}' for i in range(actual_k)]

fig, ax = plt.subplots(figsize=(9, actual_k * 0.65 + 1.5))
sns.heatmap(
    heatmap_data,
    xticklabels=ALL_CATS,
    yticklabels=yticklabels,
    cmap='YlOrRd', annot=True, fmt='.3f',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Mean component activation', 'shrink': 0.8},
)
ax.set_title('Mean BNMF Component Activation per Sound Category\n'
             '(Rows = sound concepts discovered; Columns = original 5 labels)',
             fontsize=11, pad=12)
ax.set_xlabel('Sound Category (original labels)')
ax.set_ylabel('BNMF Component (discovered)')
plt.tight_layout()
plt.savefig(FIG_DIR / "bnmf_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Category composition of top-50 clips per component ───────────────────────
ALL_CATS = ['speech', 'environment', 'nature', 'animal', 'music']

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=False)
axes_flat = axes.flatten()

for c in range(actual_k):
    top_idx = np.argsort(cons_resp[:, c])[::-1][:50]
    counts  = pd.Series(labels_cat[top_idx]).value_counts().reindex(ALL_CATS, fill_value=0)
    ax = axes_flat[c]
    ax.bar(counts.index, counts.values,
           color=[CAT_COLORS.get(x, 'gray') for x in counts.index],
           edgecolor='white', linewidth=0.5)
    ax.set_title(f'C{c+1}: {COMP_LABELS.get(c+1, "")}',
                 fontsize=9.5, fontweight='bold', color=BNMF_COLORS[c])
    ax.set_ylabel('Count' if c % 5 == 0 else '')
    ax.tick_params(axis='x', rotation=40, labelsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Category Composition of Top-50 Clips per BNMF Component\n'
             '(Which original sound categories does each discovered component capture?)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "bnmf_component_categories.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── t-SNE colored by dominant component  +  activation violin ────────────────
# reuse ast_tsne from Section 4; recompute if running this section standalone
try:
    _ = ast_tsne
except NameError:
    _sc  = StandardScaler().fit_transform(ast_feats)
    _pca = PCA(n_components=50, random_state=42).fit_transform(_sc)
    ast_tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42).fit_transform(_pca)

dominant_comp = np.argmax(cons_resp, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: t-SNE by dominant component
for c in range(actual_k):
    mask = dominant_comp == c
    axes[0].scatter(ast_tsne[mask, 0], ast_tsne[mask, 1],
                    color=BNMF_COLORS[c],
                    label=f'C{c+1}: {COMP_LABELS.get(c+1, "")}',
                    alpha=0.65, s=14, edgecolors='none')
axes[0].legend(markerscale=2.5, fontsize=7.5, loc='best', ncol=2, framealpha=0.8)
axes[0].set_title('t-SNE of AST Embeddings\nColored by Dominant BNMF Component', fontsize=10)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Right: violin — per-category activation for each component
df_act  = pd.DataFrame(cons_resp, columns=[f'C{i+1}' for i in range(actual_k)])
df_act['category'] = labels_cat
df_long = df_act.melt(id_vars='category', var_name='component', value_name='activation')

sns.violinplot(data=df_long, x='component', y='activation', hue='category',
               palette=CAT_COLORS, ax=axes[1],
               density_norm='width', inner=None, cut=0)
axes[1].set_title('Component Activation Distribution\nby Sound Category', fontsize=10)
axes[1].set_xlabel('Component'); axes[1].set_ylabel('Activation')
axes[1].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, title='Category')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('BNMF Component Structure in Embedding Space', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "bnmf_tsne_violin.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Top 3 captions per component — readable summary ──────────────────────────
df_caps_rows = []
for c in range(actual_k):
    top_idx = np.argsort(cons_resp[:, c])[::-1][:3]
    for rank, idx in enumerate(top_idx, 1):
        cap = labeled['answer'].iloc[idx]
        df_caps_rows.append({
            'Component': f'C{c+1}: {COMP_LABELS.get(c+1, "")}',
            'Rank': rank,
            'Category': labels_cat[idx],
            'Caption (top-activating clips)': cap[:90] + ('…' if len(cap) > 90 else ''),
        })

df_caps = pd.DataFrame(df_caps_rows).set_index(['Component', 'Rank'])
print("Top 3 audio clips per BNMF component (highest activation score):\n")
df_caps

## Section 8: Summary and Biological Interpretation (RQ4)
### Connecting computational results to infant auditory development

In [ ]:
summary = {
    "RQ1 — Auditory structure from naturalistic audio": {
        "Finding": "AST embeddings support high-accuracy speech/non-speech and category classification",
        "Biological parallel": "Infants discriminate speech from non-speech from birth (Vouloumanos & Werker 2007)"
    },
    "RQ2 — Sound mix shapes representations": {
        "Finding": "Speech-dominant training improves speech detection; balanced mix best for categorization",
        "Biological parallel": "Infants in speech-rich environments show faster phonetic development (Kuhl 2004)"
    },
    "RQ3 — Developmental milestones": {
        "Finding": "Speech detection reaches plateau earlier than multi-class categorization",
        "Biological parallel": "Speech/non-speech sensitivity precedes fine-grained phonetic discrimination"
    },
    "RQ4 — Biological plausibility": {
        "Finding": "AST > MFCC on all tasks; UMAP shows speech cluster separated from non-speech",
        "Biological parallel": "Deep representations resemble higher auditory cortex more than early acoustic features"
    },
}

print("=" * 70)
print("CAPSTONE RESULTS SUMMARY")
print("=" * 70)
for rq, content in summary.items():
    print(f"\n{rq}")
    print(f"  Finding:            {content['Finding']}")
    print(f"  Biological parallel:{content['Biological parallel']}")

print("\n\nKey Conclusions:")
print("1. Naturalistic audio (AudioCaps) → meaningful auditory representations emerge")
print("2. Pretrained AST dramatically outperforms hand-crafted MFCCs on all tasks")
print("3. Developmental ordering: broad categories before fine categories")
print("4. Environment matters: speech-rich input → stronger speech representations")
print("\nFigures saved to results/figures/")